# Маскирование голов: результаты нового конвейера

Читает папку перебора `rh.mask_sweep` (по подпапке на прогон: `none`, `top20`, `random20`, `groups20`, …) и папку детекции. Модель и GPU не нужны. Прогоны, которых ещё нет, просто не попадают на графики.

In [ ]:
import glob
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

## Настройки

`limit` — оценивать ответы обрезанными на этом числе токенов (`None` — полные ответы, до предела прогона).

In [ ]:
mask_name = 'qwen3_mask'      # папка перебора в results/new
detect_name = 'qwen3_detect'  # папка детекции в results/new
limit = None

# ноутбук может лежать в source/viz или запускаться из source либо из корня репозитория
root = next(p for p in ('../../results/new', '../results/new', 'results/new') if os.path.isdir(p))
mask_dir, detect_dir = f'{root}/{mask_name}', f'{root}/{detect_name}'
out_dir = 'viz' if os.path.isdir('viz') else '.'

# чёткость: в ноутбуке и в сохранённых файлах
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 220, 'font.size': 11})

kind_names = {'none': 'без маски', 'top': 'лучшие', 'random': 'случайные россыпью', 'random_groups': 'случайные группами'}
# цвет закреплён за видом маски
colors = {'top': '#eb6834', 'random': '#2a78d6', 'random_groups': '#1baf7a', 'none': '#52514e'}
score_cmap = LinearSegmentedColormap.from_list('score', ['#f0efec', '#9ec5f4', '#2a78d6', '#0d366b'])
grid_color, ink = '#e6e5e1', '#52514e'

def style(ax):
    ax.grid(axis='y', color=grid_color, linewidth=1)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)

## Загрузка

In [ ]:
runs, rows = [], []
for folder in sorted(glob.glob(f'{mask_dir}/*/')):
    name = os.path.basename(os.path.normpath(folder))
    if not os.path.exists(f'{folder}/samples.jsonl'):
        continue
    with open(f'{folder}/samples.jsonl', encoding='utf-8') as f:
        samples = [json.loads(l) for l in f if l.strip()]
    run = json.load(open(f'{folder}/spool/run.json')) if os.path.exists(f'{folder}/spool/run.json') else {}
    block, size = run.get('block_list') or [], run.get('group_size', 1)
    runs.append({'run': name, 'kind': run.get('mask', 'none'), 'heads': len(block),
                 'groups': len({(l, h // size) for l, h in block}), 'seed': (run.get('args') or {}).get('seed', 0),
                 'cap': (run.get('args') or {}).get('max_new_tokens'), 'samples': len(samples)})
    for s in samples:
        at = s['by_limit'][str(limit)] if limit is not None else {'rouge1_recall': s['rouge1_recall'], 'success': s['success'], 'truncated': not s.get('stopped', True)}
        rows.append({'run': name, 'kind': runs[-1]['kind'], 'heads': len(block), 'length': s['context_length'], 'depth': s['depth_percent'],
                     'rouge': at['rouge1_recall'], 'success': at['success'], 'truncated': at['truncated'], 'tokens': s['n_tokens']})

runs, df = pd.DataFrame(runs), pd.DataFrame(rows)
print(f'прогонов: {len(runs)}, примеров: {len(df)}')
if runs['cap'].nunique() > 1:
    print('ВНИМАНИЕ: у прогонов разный предел длины ответа, сравнивать их нельзя:', sorted(runs['cap'].unique()))
if not (runs['kind'] == 'none').any():
    print('ВНИМАНИЕ: нет прогона без маски, эффект маски не с чем сравнить')

## Сводка

`успех` — доля примеров с ROUGE-1 recall выше 50; `обрезано` — доля ответов, не закончившихся к лимиту.

In [ ]:
summary = (df.groupby('run').agg(ROUGE=('rouge', 'mean'), успех=('success', 'mean'), обрезано=('truncated', 'mean'), токенов=('tokens', 'mean'))
             .join(runs.set_index('run')[['kind', 'heads', 'groups', 'samples']]))
summary['вид'] = summary['kind'].map(kind_names)
summary = summary.sort_values(['kind', 'heads'])[['вид', 'heads', 'groups', 'samples', 'ROUGE', 'успех', 'обрезано', 'токенов']]
summary.rename(columns={'heads': 'голов', 'groups': 'групп', 'samples': 'примеров'}).round({'ROUGE': 1, 'успех': 3, 'обрезано': 3, 'токенов': 1})

## Качество от числа отключённых голов

Главный график. Точка в нуле — прогон без маски, общая для всех линий. Полосы — 95% интервал по бутстрепу примеров.

In [ ]:
def interval(values, n=2000, seed=0):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n, len(values)))].mean(1)
    return np.percentile(means, [2.5, 97.5])

base = df[df['kind'] == 'none']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, column, label, scale in ((axes[0], 'rouge', 'Средний ROUGE-1 recall', 1), (axes[1], 'success', 'Доля успешных примеров, %', 100)):
    for kind in ('top', 'random', 'random_groups'):
        part = df[df['kind'] == kind]
        if part.empty:
            continue
        points = [(0, base[column])] if not base.empty else []
        points += [(k, g[column]) for k, g in part.groupby('heads')]
        x = [k for k, _ in points]
        y = [v.mean() * scale for _, v in points]
        lo, hi = zip(*[interval(v) * scale for _, v in points])
        ax.fill_between(x, lo, hi, color=colors[kind], alpha=0.15, linewidth=0)
        ax.plot(x, y, color=colors[kind], linewidth=2, marker='o', markersize=6, label=kind_names[kind])
    ax.set_ylim(0, 105)
    ax.set_xlabel('Отключено голов')
    ax.set_ylabel(label)
    style(ax)
axes[0].legend(frameon=False, loc='lower left')
fig.suptitle(f'{mask_name}: эффект маскирования голов')
fig.tight_layout()
fig.savefig(f'{out_dir}/{mask_name}_by_heads.png')
plt.show()

## Качество по длине контекста

По панели на каждое число отключённых голов, по линии на вид маски; среднее по глубинам. Пунктир — прогон без маски, он одинаков на всех панелях.

In [ ]:
ks = sorted(df.loc[df['heads'] > 0, 'heads'].unique())   # все числа отключённых голов, для которых есть прогоны

def panels(column, xlabel, filename):
    cols = min(2, len(ks))   # две панели в ряд: так каждая достаточно крупная
    rows_n = -(-len(ks) // cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(8.5 * cols, 4.8 * rows_n), squeeze=False, sharex=True, sharey=True)
    for ax, k in zip(axes.ravel(), ks):
        for kind in ('none', 'top', 'random', 'random_groups'):
            part = df[(df['kind'] == kind) & ((df['heads'] == k) | (kind == 'none'))]
            if part.empty:
                continue
            line = part.groupby(column)['rouge'].mean()
            ax.plot(line.index, line.values, color=colors[kind], linewidth=2, marker='o', markersize=5,
                    linestyle='--' if kind == 'none' else '-', label=kind_names[kind])
        ax.set_ylim(0, 105)
        ax.set_title(f'{k} голов')
        style(ax)
    for ax in axes.ravel()[len(ks):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel(xlabel)
    for ax in axes[:, 0]:
        ax.set_ylabel('Средний ROUGE-1 recall')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(f'{mask_name}', y=1.04)
    fig.tight_layout()
    fig.savefig(f'{out_dir}/{mask_name}_{filename}.png', bbox_inches='tight')
    plt.show()

panels('length', 'Длина контекста, токенов', 'by_length')
df.pivot_table(values='rouge', index='length', columns='run', aggfunc='mean').round(1)

## Качество по глубине иглы

То же по глубинам: среднее по длинам на каждой глубине. Показывает, где маска действует: если поиск иглы в начале и в конце контекста не страдает, а в середине страдает, линия лучших голов провисает посередине.

In [ ]:
panels('depth', 'Глубина иглы, % контекста', 'by_depth')
df.pivot_table(values='rouge', index='depth', columns='run', aggfunc='mean').round(1)

## Карты «длина × глубина»

Все прогоны сразу: строка — число отключённых голов, столбец — вид маски; отдельно сверху — прогон без маски. По горизонтали длина контекста, по вертикали глубина иглы. Игла стоит на заданной глубине (в отличие от прогонов на коде авторов), так что строки карты можно читать как положение иглы в контексте.

In [ ]:
def draw_map(ax, name):
    part = df[df['run'] == name]
    table = part.pivot_table(values='rouge', index='depth', columns='length', aggfunc='mean')
    image = ax.imshow(table.values, aspect='auto', cmap=score_cmap, vmin=0, vmax=100)
    ax.set_xticks(range(0, len(table.columns), 2), table.columns[::2], rotation=45)
    ax.set_yticks(range(len(table.index)), table.index)
    ax.set_title(f'{name}: {part["rouge"].mean():.1f}', fontsize=12)
    return image

names = set(df['run'])
if 'none' in names:
    fig, ax = plt.subplots(figsize=(9, 4.2))
    image = draw_map(ax, 'none')
    ax.set_xlabel('Длина контекста, токенов')
    ax.set_ylabel('Глубина иглы, %')
    fig.colorbar(image, ax=ax, label='ROUGE-1 recall')
    fig.tight_layout()
    plt.show()

kinds = [('top', 'top', 'лучшие'), ('random', 'random', 'случайные россыпью'), ('random_groups', 'groups', 'случайные группами')]
fig, axes = plt.subplots(len(ks), len(kinds), figsize=(7.5 * len(kinds), 3.8 * len(ks)), squeeze=False, sharex=True, sharey=True)
for r, k in enumerate(ks):
    for c, (kind, prefix, title) in enumerate(kinds):
        ax = axes[r, c]
        # имя папки может нести seed (random20_s1): берём первый прогон этого вида и числа голов
        found = runs[(runs['kind'] == kind) & (runs['heads'] == k)]['run']
        if found.empty:
            ax.set_axis_off()
            ax.set_title(f'{prefix}{k}: нет прогона', fontsize=10, color=ink)
            continue
        image = draw_map(ax, found.iloc[0])
for ax, (_, _, title) in zip(axes[0], kinds):
    ax.annotate(title, xy=(0.5, 1.28), xycoords='axes fraction', ha='center', fontsize=12)
for ax in axes[:, 0]:
    ax.set_ylabel('Глубина, %')
for ax in axes[-1]:
    ax.set_xlabel('Длина контекста, токенов')
fig.colorbar(image, ax=axes, label='ROUGE-1 recall', fraction=0.02, pad=0.02)
fig.savefig(f'{out_dir}/{mask_name}_maps.png', bbox_inches='tight')
plt.show()

## Головы из детекции

Средний скор копирования каждой головы по успешным примерам. Вертикальные линии разделяют группы голов с общими ключами и значениями.

In [ ]:
scores = json.load(open(f'{detect_dir}/head_score_copy_count.json'))
run = json.load(open(f'{detect_dir}/spool/run.json')) if os.path.exists(f'{detect_dir}/spool/run.json') else {}
size = run.get('group_size', 1)
mean = {tuple(int(i) for i in key.split('-')): float(np.mean(v)) if len(v) else 0.0 for key, v in scores.items()}
layers, heads = sorted({l for l, _ in mean}), sorted({h for _, h in mean})
grid = np.array([[mean[(l, h)] for h in heads] for l in layers])

fig, ax = plt.subplots(figsize=(14, 9))
image = ax.imshow(grid, aspect='auto', cmap=score_cmap, vmin=0)
for boundary in range(size, len(heads), size):
    ax.axvline(boundary - 0.5, color='white', linewidth=1.5)
ax.set_xticks(range(len(heads)), heads)
ax.set_yticks(range(len(layers)), layers)
ax.set_xlabel('Голова')
ax.set_ylabel('Слой')
ax.set_title(f'{detect_name}: скор копирования, голов выше 0,1 — {int((grid > 0.1).sum())} из {grid.size}')
fig.colorbar(image, ax=ax, label='Средний скор', pad=0.01)
fig.tight_layout()
fig.savefig(f'{out_dir}/{detect_name}_heads.png')
plt.show()

### Сколько голов с каким скором

Три группы по среднему скору копирования: ниже 0,1 (не retrieval-головы по порогу статьи), от 0,1 до 0,5 и от 0,5.

In [ ]:
bins = [('ниже 0,1', grid < 0.1, '#d9d8d3'), ('0,1 – 0,5', (grid >= 0.1) & (grid < 0.5), '#6da7ec'), ('0,5 и выше', grid >= 0.5, '#0d366b')]
counts = [int(mask.sum()) for _, mask, _ in bins]
labels = [f'{name}: {n} голов, {n / grid.size:.1%}' for (name, _, _), n in zip(bins, counts)]

# подписи вынесены в легенду: два малых сектора стоят рядом, и подписи у самих секторов наезжают друг на друга
fig, ax = plt.subplots(figsize=(9, 6))
wedges, _ = ax.pie(counts, colors=[color for _, _, color in bins], startangle=90, counterclock=False,
                   wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.legend(wedges, labels, loc='center left', bbox_to_anchor=(0.95, 0.5), frameon=False)
ax.set_title(f'{detect_name}: головы по скору копирования, всего {grid.size}')
fig.tight_layout()
fig.savefig(f'{out_dir}/{detect_name}_heads_pie.png')
plt.show()

pd.DataFrame({'скор': [name for name, _, _ in bins], 'голов': counts, 'доля, %': [round(100 * n / grid.size, 1) for n in counts]})

### Лучшие головы и их группы

`в группе из лучших` — сколько голов той же группы входит в тот же список лучших, включая саму голову.

In [ ]:
top_n = 50
ranked = sorted(mean, key=lambda key: -mean[key])[:top_n]
in_group = {key: sum(1 for l, h in ranked if l == key[0] and h // size == key[1] // size) for key in ranked}
table = pd.DataFrame([{'место': i + 1, 'слой': l, 'голова': h, 'группа': f'{l}:{h // size * size}–{h // size * size + size - 1}',
                       'скор': round(mean[(l, h)], 3), 'в группе из лучших': in_group[(l, h)]} for i, (l, h) in enumerate(ranked)])
groups = len({(l, h // size) for l, h in ranked})
print(f'лучшие {top_n} голов лежат в {groups} группах; в одиночку в своей группе — {sum(v == 1 for v in in_group.values())}')
table.head(20)